# Demonstration Notebook
This notebook demonstrates and compares three ABSA implementations:
1. Lexicon-Based ABSA (spaCy + Sentiment Lexicon)
2. Transformer-Based ABSA (Pretrained Transformer Model)
3. LLM-Based ABSA using Ollama (Local LLM)

**The goal is to:**
- Run all three models on sample review data
- Compare their outputs on identical inputs
- Measure accuracy and runtime
- Analyze strengths, weaknesses, and trade-offs


In [ ]:
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.lexicon_absa import LexiconABSA
from src.transformer_absa import TransformerABSA
from src.llm_absa import LLMABSA

## 1. Load and Preview Data

In [ ]:
# We'll use a subset of Yelp reviews as the evaluation dataset.
# Each review includes 'text' and 'stars' fields (1–5 rating scale).

df = pd.read_json("../data/yelp_sample_20k.json", lines=True)
df.head()

## 2. Sample a Subset for Evaluation

In [ ]:
# To keep runtime manageable, we'll sample 20 reviews randomly.
# This allows a fair comparison while still showing representative examples.

sample_df = df.sample(20, random_state=42).reset_index(drop=True)
sample_df.head()

## 3. Initialize All Three ABSA Implementations

In [ ]:
# Initialize each ABSA model.
# Make sure spaCy model and Ollama LLM are properly installed before running.

lex_model = LexiconABSA()
trf_model = TransformerABSA()
llm_model = LLMABSA("llama3")  # Ensure Ollama is running locally

## 4. Run ABSA Across All Models

In [ ]:
# We'll process each review with all three models.
# For each model, we'll store:
#   • Extracted (aspect, sentiment) pairs
#   • Processing time

results = []

for i, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    text = row["text"]
    row_result = {"review": text}

    for model_name, model in [
        ("LexiconABSA", lex_model),
        ("TransformerABSA", trf_model),
        ("LLMABSA", llm_model),
    ]:
        start = time.time()
        aspects = model.analyze(text)
        end = time.time()

        row_result[f"{model_name}_time"] = round(end - start, 3)
        row_result[f"{model_name}_aspects"] = [(a.aspect, a.sentiment) for a in aspects]

    results.append(row_result)

comp_df = pd.DataFrame(results)
comp_df.head()

## 5. Interactive Comparison Function

In [ ]:
# Function to display model comparisons for a given review
def show_comparison(idx):
    row = comp_df.iloc[idx]
    print(f"Review {idx+1}: {row['review'][:1000]}...\n")
    for model_name in ["LexiconABSA", "TransformerABSA", "LLMABSA"]:
        print(f"--- {model_name} ---")
        aspects = row[f"{model_name}_aspects"]
        if not aspects:
            print("No aspects detected.")
        else:
            for asp, sent in aspects:
                print(f"   ➜ {asp:<25} | {sent}")
        print(f"Runtime: {row[f'{model_name}_time']}s\n")


In [ ]:
# Example comparisons (adjust indices to get a desired review)
show_comparison(0)

In [ ]:
show_comparison(5)

In [ ]:
show_comparison(10)

## 6. Compute Approximate Accuracy

In [ ]:
# We'll estimate sentiment accuracy using Yelp stars as a proxy label.
# This is a coarse measure but useful for trend analysis.

def yelp_to_sentiment(stars):
    if stars >= 4:
        return "positive"
    elif stars == 3:
        return "neutral"
    else:
        return "negative"

sample_df["true_sentiment"] = sample_df["stars"].apply(yelp_to_sentiment)

def aggregate_sentiment(preds):
    if not preds:
        return "neutral"
    pos = sum(1 for _, s in preds if s == "positive")
    neg = sum(1 for _, s in preds if s == "negative")
    if pos > neg:
        return "positive"
    elif neg > pos:
        return "negative"
    return "neutral"


## 7. Compute Metrics: Accuracy, Runtime, and Aspects

In [ ]:
# Compute overall accuracy and average runtime per model.
# Also compute average number of aspects detected per review

scores = []
for model_name in ["LexiconABSA", "TransformerABSA", "LLMABSA"]:
    pred_sentiments = [
        aggregate_sentiment(row[f"{model_name}_aspects"]) for _, row in comp_df.iterrows()
    ]
    acc = (pd.Series(pred_sentiments) == sample_df["true_sentiment"]).mean()
    avg_time = comp_df[f"{model_name}_time"].mean()
    avg_aspects = comp_df[f"{model_name}_aspects"].apply(len).mean()

    scores.append((model_name, round(acc, 3), round(avg_time, 3), round(avg_aspects, 2)))

metrics_df = pd.DataFrame(scores, columns=["Model", "Accuracy", "Avg_Runtime(s)", "Avg_Aspects"])

In [ ]:
print("ABSA Model Performance Summary")
display(metrics_df.style.set_caption("Performance Metrics Across ABSA Implementations"))


## 8.Visualize Metrics

In [ ]:
# Accuracy comparison
plt.figure(figsize=(8,5))
plt.bar(metrics_df["Model"], metrics_df["Accuracy"], color=["skyblue","orange","green"])
plt.title("Accuracy Comparison (vs. Yelp Star Ratings)")
plt.ylabel("Accuracy")
plt.show()

# Runtime comparison
plt.figure(figsize=(8,5))
plt.bar(metrics_df["Model"], metrics_df["Avg_Runtime(s)"], color=["gray","gold","lightcoral"])
plt.title("Average Runtime per Review")
plt.ylabel("Seconds")
plt.show()

# Average Aspects comparison
plt.figure(figsize=(8,5))
plt.bar(metrics_df["Model"], metrics_df["Avg_Aspects"], color=["teal","purple","coral"])
plt.title("Average Aspects Detected per Review")
plt.ylabel("Count")
plt.show()


## Results Showcase Side-by-Side Comparisons

In [ ]:
# A few qualitative examples to illustrate how each model differs.

showcase_examples = [
    """The mushroom risotto was perfection, but the service was slow.
    The atmosphere was cozy and the chocolate cake divine.""",

    """I tried their decaf latte which was tasty, foamy, and light.
    The baristas were friendly but the parking lot was always packed.""",

    """Amazing bagels! My favorite place ever. Super fast cook and donuts are fresh and taste so good!"""
]

def compare_models_on_text(text):
    rows = []
    for model_name, model in [
        ("LexiconABSA", lex_model),
        ("TransformerABSA", trf_model),
        ("LLMABSA", llm_model)
    ]:
        aspects = model.analyze(text)
        if not aspects:
            rows.append({
                "Model": model_name,
                "Aspect": "_No aspects detected_",
                "Sentiment": "",
                "Confidence": ""
            })
        else:
            for a in aspects:
                rows.append({
                    "Model": model_name,
                    "Aspect": a.aspect,
                    "Sentiment": a.sentiment,
                    "Confidence": round(a.confidence, 2)
                })
    return pd.DataFrame(rows)

from IPython.display import HTML, display
pd.set_option("display.max_colwidth", None)
display(HTML("<style>.dataframe td { white-space: pre-wrap; }</style>"))

for i, review in enumerate(showcase_examples, 1):
    print(f"### Review {i}\n> {review.strip()}\n")
    df_show = compare_models_on_text(review)
    display(df_show)
    print("\n")


# Final Conclusions